In [13]:
#### Setup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import SessionNotCreatedException
from selenium.common.exceptions import ElementNotVisibleException
from selenium.common.exceptions import TimeoutException


import csv
import time

import pandas as pd

In [14]:
#### Helper Functions
# Initialize Browser
def InitializeBrowser(start_url, webDriverPath):
    # set up web driver
    s = Service(webDriverPath)
    browser = webdriver.Chrome(service=s)
    
    # intialize browser at specified start url
    browser.get(start_url)
    
    return browser

# This function takes in a webElement and gets relevant information from the text
# i.e. the permit type and its status
def GetStatAndPermText(webElement):
    # get the inner html code
    innerHTML = webElement.get_attribute("innerHTML").split('">')
    
    # get the status and permit from the text
    stat = innerHTML[1].split('<')[0]
    perm = innerHTML[2].split('<')[0].split(' (')[0]

    # return the information
    return stat, perm

# This functions takes in a string of text and gets the number of inspection entries
def GetNumRecords(browser):
    
    try:
        # get string containing the number of completed inspections
        completedText = WebDriverWait(browser, 60).until(EC.visibility_of_element_located((By.ID, "ctl00_PlaceHolderMain_InspectionList_lblInspectionCompleted"))).text

        # eliminate the "Complete " and then the number of entries is next
        completedInsp = completedText.split('Completed ')

        # if there is no entry for the number of inspections, 0 are completed
        if (len(completedInsp) == 1):
            return 0

        # get the number of entries from inside the parentheses
        else:
            completedInsp = completedInsp[1].split('\n')
            numRecords = int(completedInsp[0][1:-1])

            # return the number of records
            return numRecords
     
    # if the inspections table does not load, say table is empty
    except (TimeoutException, ElementNotVisibleException):
        return -1

    

def OpenFiles(filenameResult="PermitStatus", filenameFailure="GetStatusFailed", filenameSuccess=None, overwrite_csv=False):

    if (filenameSuccess == None):
        keepRawPermitStatus = False
    else:
        keepRawPermitStatus = True

    if (overwrite_csv):
        # open files to write; overwrites
        fileResult = open("./"+ filenameResult +".csv", mode='w')
        fileFailure = open("./" + filenameFailure + ".csv", mode='w')

        # create csv writer for data
        writerResult  = csv.writer(fileResult)
        writerFailure = csv.writer(fileFailure)

        # add column names
        row = ["ID"]
        row = row + ["Most Recent"]
        writerResult.writerow(row)
        
        if (keepRawPermitStatus):
            fileSuccess = open("./"+ filenameSuccess +".csv", mode='w')
            row = ["ID"]
            row = row + relevant_permits
            writerSuccess = csv.writer(fileSuccess)
            writerSuccess.writerow(row)     
            
        writerFailure.writerow(["ID"])

    # open files in append mode    
    else:
        # open files to write to using; does not overwrite, just appends
        fileResult = open("./"+ filenameResult +".csv", mode='a')
        fileFailure = open("./" + filenameFailure + ".csv", mode='a')

        # create csv writer for data
        writerResult  = csv.writer(fileResult)
        if (keepRawPermitStatus):
            fileSuccess = open("./" + filenameSuccess + ".csv", mode='a')
            writerSuccess = csv.writer(fileSuccess)
        writerFailure = csv.writer(fileFailure)
    
    
    if (keepRawPermitStatus):
        files = [fileResult, fileFailure, fileSuccess]
        writers = [writerResult, writerFailure, writerSuccess]
        return files, writers
    else:
        files = [fileResult, fileFailure]
        writers = [writerResult, writerFailure]
        return files, writers
    
    
    
    
### Determine the permits that did not get recorded and write to a specified file
def GetUnsuccessfulPermits(filename, permit_list_use, filenameSuccess):

    # get the permits that were recorded
    recordedPermits = pd.read_csv("./" + filenameSuccess + ".csv")
    recordedPermits.drop(labels=recordedPermits.columns.difference(["ID"]), axis=1, inplace=True)

    # get the list of permits used
    triedPermitsList = {"permits": permit_list_use}
    triedPermits = pd.DataFrame(triedPermitsList)
    triedPermits = triedPermits.set_index(keys=triedPermits["permits"])
    triedPermits = triedPermits.rename(columns={"permits":"ID"})

    # get unused permits (due to failure)
    unusedPermits = triedPermits.loc[~triedPermits.index.isin(recordedPermits["ID"])]
    unusedPermits = unusedPermits.reset_index()
    unusedPermits.drop(columns=["permits"], inplace=True, axis=1)
    
    # write to file
    unusedPermits["ID"].to_csv(path_or_buf=filename, index=False)



In [18]:
# This function tries to move to the next page of inspections
# If it fails, it recursively calls itself to try again until success or the number of times tried reaches the "quit_count"
def RecursivePageTurn(p, browser, delay, permit_number, counter, quit_count=20):
    try:
        # CSS Selector for the given page
        cssSelStr = "#ctl00_PlaceHolderMain_InspectionList_gvListCompleted > tbody > tr.ACA_Table_Pages.ACA_Table_Pages_FontSize > td > table > tbody > tr > td:nth-child(" + str(p+2) + ")"
        
        # web element of the table of inspections
        table = browser.find_element(by=By.CSS_SELECTOR, value=cssSelStr)
        
        # click to the next page
        table.click()
        
        # delay to decrease the chance of exceptions
        time.sleep(delay)

    # if a failure occurs, try again
    except (ElementClickInterceptedException, StaleElementReferenceException) as e:
        print(f"Exception {e} {counter}! Page turn failure on page {p} on try {counter} for permit #{permit_number}")
        
        # continue to try to turn the page
        if (counter < quit_count):
            # recursively call function to continue trying to turn the page
            return RecursivePageTurn(p, browser, delay, permit_number, counter+1, quit_count) 
            
        # if too many tries for turning the page were reached, quit and return False for failure
        else:
            print(f"Exception {e}! Page turn failure on page {p} on try {counter} for permit #{permit_number}! Retry permit later!")
            return False
          
    # This exception occurs when the format of the page changes due to the list of pages 
    # (i.e. for pages 1-10 there are 12 elements, but for 11 or more, there are only 4 web elements)
    except NoSuchElementException as nsee:
        print(f"Exception {nsee}! Page turn failure on page {p} on try {counter} for permit #{permit_number}! No Such Element b/c of page formatting. Continue until found.")
        return RecursivePageTurn(p, browser, delay, permit_number, counter+1, quit_count)
        # FIX: IF THIS DOESN'T WORK, I CHANGED p-1 to p AND CHANGED counter to counter+1

    # at this point, success has occurred
    return True




# This function accesses the search box on the pages and searches for
# a given permit ID
def SearchForPermit(browser, permit_number, quit_count=10):
    # enter permit id into search box
    searchbox = browser.find_element(by=By.ID, value="txtSearchCondition")

    # clear and then enter the permit number into the search bar
    searchbox.clear()
    searchbox.send_keys(str(permit_number))

    # find and click on button to search id
    counterSearch = 1
    searchSuccess = RecursiveSearchButtonClick(browser, permit_number, counterSearch, quit_count)
    
    # return whether or not the search was successful
    return searchSuccess


# This function continually/recursively tries to click the search button
def RecursiveSearchButtonClick(browser, permit_number, counter, quit_count=10):
    # try clicking the search button
    try:
        # find the search button and click it
        permitSearchBtn = browser.find_element(by=By.CLASS_NAME, value="gs_go_img")
        permitSearchBtn.click()
        
    # if it fails, try again    
    except (ElementClickInterceptedException, StaleElementReferenceException) as e:
        print(f"Exception {e}! Failed to click search button on try {counter} for permit #{permit_number}!")
        
        # recursively call function to continue trying to click the search button
        if (counter < quit_count):
            return RecursiveSearchButtonClick(browser, permit_number, counter+1, quit_count)
        
        # if too many tries occur, quit and return False for failure
        else:
            print(f"Exception {e}! Failed to click search button for permit #{permit_number}! Retry permit later!")
            return False
        
    # at this point, success has occurred    
    return True


# This function reads each inspection table and gets each status and permit type    
def GetInspectionInfo(browser, status, permit, permit_number, quit_count):
    
    # read status and of permit inspections (this is used to get the number on each page)
#     completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")
# 
#     # get the table of completed inspections
#     inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")

    inspTable = browser.find_element(by=By.ID, value="divInspectionListCompleted").find_elements(by=By.CLASS_NAME, value="ACA_Width45em")
    lengthTable = len(inspTable)
    
    # for each completed inspection, get the permit and status
    # done in groups of 5
    counter = 1
    
    for i in range(0,lengthTable):
        RecursiveGetInspectionInfo(browser, i, lengthTable, status, permit, permit_number, counter, quit_count)
        
    # FIX AND INCORPORATE TRUE FALSE INTO MAIN CODE
    # ALSO FIX WRITING PROBLEM BY OPENING AND CLOSING FILE EACH RUN OF THE LOOP
    
        

def RecursiveGetInspectionInfo(browser, i, lengthTable, status, permit, permit_number, counter, quit_count=10):
    try: 
        # read status and of permit inspections (this is used to get the number on each page)
        completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")

        # get the table of completed inspections
        inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")

#         if type(lengthTable) is list:
#             lengthTable = lengthTable[0]
            
#         if type(i) is list:
#             i = i[0]
        
#         # if somehow, the index is greater than allowed, exit this function
#         if type(lengthTable) is str:
#             completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")
#             inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")
#             lengthTable = len(inspTable)
#             print(f"STRING LENGTH TABLE: {lengthTable} with type: {type(lengthTable)}")
            
#         if (i >= lengthTable):
#             return None

        
        try: 
            # get the inner html code
            innerHTML = inspTable[i].get_attribute("innerHTML").split('">')
        except IndexError as e:
            print(f"Index Error!")
            return None

        # get the status and permit from the text
        statusText = innerHTML[1].split('<')[0]
        permitText = innerHTML[2].split('<')[0].split(' (')[0]

        # get the status and permit
        status.append(statusText)
        permit.append(permitText)

    except (StaleElementReferenceException) as e:
        print(f"Exception {e}! Failed to get inspection data from table on try {counter} for permit #{permit_number}!")

        # continue trying to grab info from the table
        if (counter < quit_count):
            return RecursiveGetInspectionInfo(browser, i, lengthTable, status, permit, permit_number, counter, quit_count)   

        else: 
            print(f"Exception {e}! Failed to get inspection data from table on try {counter} for permit #{permit_number}! Retry permit later!")
            return False

    return True
        
    


# This function goes to the inspections page/tab
def GoToInspections(browser):
    # Go to Inspections for the searched for permit ID
    RecInfoDropdown = browser.find_element(by=By.CSS_SELECTOR, value="[title^='Record Info']")
    RecInfoDropdown.click()

    # keep dropdown open in order to click on inspections option
    browser.implicitly_wait(1)

    # click on "Inspection" option and go to inspections page for given permit id
    InspDropdownBtn = browser.find_element(by=By.CSS_SELECTOR, value="[title^='Inspections']")
    InspDropdownBtn.click()

    
    
def GetInspectionStatus(status, permit, permit_number):
    # put status and permit into a pandas dataframe
    data = {"status":status, "permit":permit}
    statusPanda = pd.DataFrame(data)

    # determine status of permit
    passed = statusPanda["status"] == "Pass"

    # begin row to write in csv with permit number
    row = [permit_number]

    # for each relevant permit
    for per in relevant_permits:
        # check if the permit type is in the list
        permitType = statusPanda["permit"] == per

        # if the specified permit type did not pass (resulting dataframe is empty) use "N" for no
        if (statusPanda.loc[((passed) & (permitType))].empty):
            row.append("N")
        # if the specified permit type passed use "Y" for yes
        else:
            row.append("Y")
                 
    return row  


        

In [19]:
#### Scrape Data
def ScrapeData(permit_list, state, relevant_permits, webDriverPath, overwrite_csv=False, filenameResult="PermitStatus", filenameSuccess="GetStatusSuccess", keepRawPermitStatus=True, filenameFailure="GetStatusFailed", numTryClick=20):
    # get start time
    start_time = time.time()

    if type(permit_list) is not list:
        permit_list = [permit_list]

    # open files 
    if (overwrite_csv):
        if (keepRawPermitStatus):
            # open files and initialize writers; use raw data file
            [fileResult, fileFailure, fileSuccess], [writerResult, writerFailure, writerSuccess] = OpenFiles(filenameResult=filenameResult, filenameFailure=filenameFailure, filenameSuccess=filenameSuccess, overwrite_csv=overwrite_csv)
            fileResult.close(); fileFailure.close(); fileSuccess.close();
        else:
            # open files and initialize writers; don't use raw data file
            [fileResult, fileFailure], [writerResult, writerFailure] = OpenFiles(filenameResult=filenameResult, filenameFailure=filenameFailure, overwrite_csv=overwrite_csv)
            fileResult.close(); fileFailure.close();

    # go to start url
    start_url = 'https://secureapps.charlottecountyfl.gov/CitizenAccess/Welcome.aspx?TabName=Home&TabList=Home'    

    # initialize browser
    browser = InitializeBrowser(start_url, webDriverPath)


    # NOTE: IF YOU GET SessionNotCreatedException b/c you have a different
    # chromedriver version than your chrome version,
    # Go to: https://chromedriver.storage.googleapis.com/index.html
    # to download a chromedriver version compatible with your current
    # Chrome version.
    # The error message will say something like: 
    # "Message: session not created: This version of ChromeDriver only supports Chrome version 101
    # Current browser version is 103.0.5060.114"

    success = True

    # for each permit id
    for numIter, permit_number in enumerate(permit_list):
        
        # open files for appending
        if (keepRawPermitStatus):
            # open files and initialize writers; use raw data file
            [fileResult, fileFailure, fileSuccess], [writerResult, writerFailure, writerSuccess] = OpenFiles(filenameResult=filenameResult, filenameFailure=filenameFailure, filenameSuccess=filenameSuccess, overwrite_csv=False)
        else:
            # open files and initialize writers; don't use raw data file
            [fileResult, fileFailure], [writerResult, writerFailure] = OpenFiles(filenameResult=filenameResult, filenameFailure=filenameFailure, overwrite_csv=False)

        # search for given permit id on website
        searchSuccess = SearchForPermit(browser=browser, permit_number=permit_number, quit_count=numTryClick)

        # if clicking the search button was a success
        if (searchSuccess == True):

            # Go to inspections page/tab
            GoToInspections(browser)

            # get number of records and number of pages
            numRecords = GetNumRecords(browser)

            # if numRecords==-1, the inspection table did not load and couldn't be read
            if (numRecords == -1):
                # move to the next permit
                continue;

            numPages = int(numRecords/5) + 1

            # time delay for preventing stale reference
            delay = 1.5

            # store each completed permit and status
            status = list()
            permit = list()

            # for each page   
            for p in range(1, numPages+1):

                # stop at 10 pages
#                 if (p > 10):
#                     break;
                
                #####
                # Reads each inspection page and get each status and permit type 
                getInfoSuccess = GetInspectionInfo(browser, status, permit, permit_number, numTryClick)
                #####
    
                ##############################
                # read status and of permit inspections (this is used to get the number on each page)
#                 completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")
#                 
                # get the table of completed inspections
#                 inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")
#                 inspTable = browser.find_element(by=By.ID, value="divInspectionListCompleted").find_elements(by=By.CLASS_NAME, value="ACA_Width45em")
                
#                 # for each element in the table
#                 for i in range(0,len(inspTable)):
#                     # read status and of permit inspections (this is used to get the number on each page)
# #                     completed = browser.find_element(by=By.ID, value="divInspectionListCompleted")
# # 
#                     # get the table of completed inspections
# #                     inspTable = completed.find_elements(by=By.CLASS_NAME, value="ACA_Width45em")
#                     inspTable = browser.find_element(by=By.ID, value="divInspectionListCompleted").find_elements(by=By.CLASS_NAME, value="ACA_Width45em")


#                     # get the inner html code
#                     innerHTML = inspTable[i].get_attribute("innerHTML").split('">')

#                     # get the status and permit from the text
#                     statusText = innerHTML[1].split('<')[0]
#                     permitText = innerHTML[2].split('<')[0].split(' (')[0]

#                     # get the status and permit
#                     status.append(statusText)
#                     permit.append(permitText)
                ###############################
                
                # initialize counter for counting the number of tries to turn the page
                counter = 1

                # set success equal to True for if the if statement conditions are not met
                success = True

                # if there is only one page (numRecords > 5) and the current page is not the last, turn the page
                if ((numRecords > 5) & (p != numPages)):
                    success = RecursivePageTurn(p, browser, delay, permit_number, counter, quit_count=numTryClick)


            # if the permit information was successfully retrieved
            if (success == True):
                # get the status of each permit inspection
                row = GetInspectionStatus(status, permit, permit_number)
                
                rowResult = [permit_number]
                
                # get the most recent permit
                found = False
                ind = []
                for i in range(1, len(row))[::-1]:
                    if (row[i]=="Y"):
                        rowResult = rowResult + [relevant_permits[i-1]]
                        found = True
                        break;
                
                # if no relevant inspections have been completed
                if (found == False):
                    rowResult = rowResult + ["None"]

                # write a row to the csv
                writerResult.writerow(rowResult)
                
                # store raw csv data if specified
                if (keepRawPermitStatus):
                    writerSuccess.writerow(row)
                
            # if the permit information was not successfully retrieved
            else:
                writerFailure.writerow([permit_number])
        else:
            writerFailure.writerow([permit_number])
        
        # close the csv files
        fileResult.close()
        fileFailure.close()

        if (keepRawPermitStatus):
            fileSuccess.close()

    # record all the permits that were unsuccessful due to errors and exceptions
#     GetUnsuccessfulPermits("UnsuccessfulPermits", permit_list, filenameSuccess)

    # get ending time
    end_time = time.time()

    # print execution time, number of permits checked,
    exe_time = end_time - start_time
#     print(f"{exe_time} seconds to execute.")
#     print(f"{permit_number} on iteration {numIter}")
    return status, permit, row

In [20]:
# read data
data=pd.read_csv("./sampledata-permits.csv")

# list of permits
data_dict=data.to_dict()
permit_list=data["PermitNumber"].to_list() #displays all permits
length=len(data.columns)

# state of inspection
# note that "Not Started "
state = ["Not Started (NS)", "Slab", "Wall", "Roof", "Inspection (INSP)"]

# relevant permits
relevant_permits = ["Footer", "Slab", "Wall Sheathing", "Roof Sheathing", 
                    "Dry In", "Electric Rough", "Framing", "Electric Temporary Service",
                    "Insulation"]

# path to webdriver
webDriverPath = "/Users/alexanderwozny/Documents/chromedriver"

# scrape data
permit_use = permit_list[0:int(0.04*len(permit_list))+1]
# permit_use = [permit_list[3]]
status, permit, row = ScrapeData(permit_use, state, relevant_permits, webDriver, overwrite_csv=True)

Exception Message: stale element reference: element is not attached to the page document
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000102aac079 chromedriver + 4444281
1   chromedriver                        0x0000000102a38403 chromedriver + 3970051
2   chromedriver                        0x00000001026d3038 chromedriver + 409656
3   chromedriver                        0x00000001026d5ee7 chromedriver + 421607
4   chromedriver                        0x00000001026d5d86 chromedriver + 421254
5   chromedriver                        0x00000001026d604c chromedriver + 421964
6   chromedriver                        0x00000001027107c6 chromedriver + 661446
7   chromedriver                        0x000000010270e593 chromedriver + 652691
8   chromedriver                        0x000000010270bc24 chromedriver + 642084
9   chromedriver                        0x000000010270a885 chromedriver + 637061
10  chromedriver                        0x00000

Exception Message: stale element reference: element is not attached to the page document
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000102aac079 chromedriver + 4444281
1   chromedriver                        0x0000000102a38403 chromedriver + 3970051
2   chromedriver                        0x00000001026d3038 chromedriver + 409656
3   chromedriver                        0x00000001026d5ee7 chromedriver + 421607
4   chromedriver                        0x00000001026d5d86 chromedriver + 421254
5   chromedriver                        0x00000001026d604c chromedriver + 421964
6   chromedriver                        0x00000001027107c6 chromedriver + 661446
7   chromedriver                        0x000000010270e593 chromedriver + 652691
8   chromedriver                        0x000000010270bc24 chromedriver + 642084
9   chromedriver                        0x000000010270a885 chromedriver + 637061
10  chromedriver                        0x00000

Exception Message: stale element reference: element is not attached to the page document
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000102aac079 chromedriver + 4444281
1   chromedriver                        0x0000000102a38403 chromedriver + 3970051
2   chromedriver                        0x00000001026d3038 chromedriver + 409656
3   chromedriver                        0x00000001026d5ee7 chromedriver + 421607
4   chromedriver                        0x00000001026d5d86 chromedriver + 421254
5   chromedriver                        0x00000001026d604c chromedriver + 421964
6   chromedriver                        0x00000001027107c6 chromedriver + 661446
7   chromedriver                        0x000000010270e593 chromedriver + 652691
8   chromedriver                        0x000000010270bc24 chromedriver + 642084
9   chromedriver                        0x000000010270a885 chromedriver + 637061
10  chromedriver                        0x00000

StaleElementReferenceException: Message: stale element reference: element is not attached to the page document
  (Session info: chrome=103.0.5060.114)
Stacktrace:
0   chromedriver                        0x0000000102aac079 chromedriver + 4444281
1   chromedriver                        0x0000000102a38403 chromedriver + 3970051
2   chromedriver                        0x00000001026d3038 chromedriver + 409656
3   chromedriver                        0x00000001026d5ee7 chromedriver + 421607
4   chromedriver                        0x00000001026d5d86 chromedriver + 421254
5   chromedriver                        0x00000001026d604c chromedriver + 421964
6   chromedriver                        0x00000001027090d5 chromedriver + 630997
7   chromedriver                        0x0000000102709581 chromedriver + 632193
8   chromedriver                        0x00000001026fe524 chromedriver + 587044
9   chromedriver                        0x00000001027267bd chromedriver + 751549
10  chromedriver         

In [12]:
for i, val in enumerate(permit_list):
    if (val == 20210933913):
        print(i)
        break;

35


In [57]:
len(permit_list)*0.04

41.160000000000004

In [154]:
for i in range(0, len(status)):
    print(f"{status[i]}\t{permit[i]}")

Pass	Plumbing Underground
Pass	Slab
Pass	Bond Beam
Pass	Buck
Pass	Fill Cells
Pass	Roof Sheathing
Pass	Tie Down
Pass	Dry In
Pass	ROW Line and Grade
Pass	HVAC Rough
Pass	ROW Pipe
Pass	Plumbing Rough
Pass	Roof Final
Pass	Stucco/Lath
Pass	Electric Rough
Pass	Fire Wall
Pass	Framing
Fail	Electric Temporary Service
Partial Without Fee	Insulation
Pass	Soffit
Pass	Plumbing Sewer/Septic
Fail without Fee	Electric Temporary Service
Pass	Electric Temporary Service
Approved as Noted	Plans Change Submitted


In [155]:
# put status and permit into a pandas dataframe
data = {"status":status, "permit":permit}
statusPanda = pd.DataFrame(data)
statusPanda

,status,permit
0,Pass,Plumbing Underground
1,Pass,Slab
2,Pass,Bond Beam
3,Pass,Buck
4,Pass,Fill Cells
5,Pass,Roof Sheathing
6,Pass,Tie Down
7,Pass,Dry In
8,Pass,ROW Line and Grade
9,Pass,HVAC Rough


In [156]:
passed = statusPanda["status"] == "Pass"
permit_type = statusPanda["permit"] == relevant_permits[1]
statusPanda.loc[(passed & permit_type)].empty

False

In [4]:
thing = [1, 2, 3]
[thing1, thing2, thing3] = thing

In [ ]:
# determine status of permit
passed = statusPanda["status"] == "Pass"

# begin row to write in csv with permit number
row = [404]

# for each relevant permit
for per in relevant_permits:
    # check if the permit type is in the list
    permitType = statusPanda["permit"] == per

    # if the specified permit type did not pass (resulting dataframe is empty) use "N" for no
    if (statusPanda.loc[((passed) & (permitType))].empty):
        row.append("N")
    # if the specified permit type passed use "Y" for yes
    else:
        row.append("Y")
 